In [8]:
import os
import http.server
import socketserver
import threading
import IPython.display as display
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import numpy as np
from plotly.subplots import make_subplots

print("⏳ Building Publication-Style Dashboard (No Statsmodels required)...")

# ==========================================
# 1. GENERATE DATA & REGRESSION
# ==========================================
np.random.seed(42)
n_points = 80

# --- Panel A: PCA Scree ---
pca_var = [32, 9, 6, 4, 3.5, 3, 2.5, 2, 1.5, 1]
pca_x = [f"PC{i}" for i in range(1, 11)]

# --- Scatter Data & Manual Regression ---
x_gaf = np.random.normal(0, 20, n_points)
y_panss = -0.6 * x_gaf + np.random.normal(0, 10, n_points)
x_panss = np.random.normal(0, 20, n_points)

def get_regression_line(x, y):
    m, b = np.polyfit(x, y, 1)
    x_line = np.linspace(min(x), max(x), 100)
    y_line = m * x_line + b
    return x_line, y_line

x_line_b, y_line_b = get_regression_line(x_gaf, y_panss)
x_line_c, y_line_c = get_regression_line(x_gaf, np.random.normal(0, 10, n_points))
x_line_d, y_line_d = get_regression_line(x_panss, np.random.normal(0, 10, n_points))

# --- Panel E: Topographies ---
def create_head_grid():
    x, y = np.linspace(-1, 1, 100), np.linspace(-1, 1, 100)
    X, Y = np.meshgrid(x, y)
    mask = X**2 + Y**2 <= 1
    return X, Y, mask

X, Y, head_mask = create_head_grid()

def get_topo(z_data):
    z_data[~head_mask] = np.nan
    return go.Heatmap(x=X[0], y=Y[:,0], z=z_data, colorscale='RdBu_r', showscale=False, zmin=-1, zmax=1)

z1 = np.exp(-((X - 0.5)**2 + Y**2) / 0.1) - np.exp(-((X + 0.5)**2 + Y**2) / 0.1)
z2 = -np.exp(-((X - 0.5)**2 + Y**2) / 0.1) + np.exp(-((X + 0.5)**2 + Y**2) / 0.1)
z3 = np.exp(-(X**2 + (Y - 0.6)**2) / 0.1) - np.exp(-(X**2 + (Y + 0.6)**2) / 0.1)
z4 = -np.exp(-(X**2 + (Y - 0.6)**2) / 0.1) + np.exp(-(X**2 + (Y + 0.6)**2) / 0.1)

# ==========================================
# 2. BUILD THE PLOTLY FIGURES
# ==========================================
layout_update = dict(template="plotly_dark", paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', margin=dict(l=40, r=20, t=40, b=40))

# A: Scree
fig_a = go.Figure(go.Scatter(x=pca_x, y=pca_var, mode='lines+markers', line=dict(color='#f472b6', width=3), marker=dict(size=8, color='#38bdf8')))
fig_a.update_layout(**layout_update, title="A: PCA Scree Plot", xaxis_title="Component", yaxis_title="Explained Variance (%)")

# B, C, D: Scatter Plots with Manual Trendlines
def build_scatter(x, y, x_line, y_line, title, color, x_label, y_label):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x, y=y, mode='markers', marker=dict(color=color, size=8, opacity=0.6), name='Data'))
    fig.add_trace(go.Scatter(x=x_line, y=y_line, mode='lines', line=dict(color='#f472b6', width=2), name='Regression'))
    fig.update_layout(**layout_update, title=title, xaxis_title=x_label, yaxis_title=y_label)
    return fig

fig_b = build_scatter(x_gaf, y_panss, x_line_b, y_line_b, "B: GAF vs PANSS (R=-0.65, p<0.001)", '#38bdf8', "V2 - V1 GAF", "V2 - V1 PANSS")
fig_c = build_scatter(x_gaf, np.random.normal(0, 10, n_points), x_line_c, y_line_c, "C: PCA Component 1 vs GAF", '#38bdf8', "V2-V1 GAF", "PCA Component 1")
fig_d = build_scatter(x_panss, np.random.normal(0, 10, n_points), x_line_d, y_line_d, "D: PCA Component 2 vs PANSS", '#c084fc', "V2-V1 PANSS", "PCA Component 2")

# E: Topographies Grid
fig_e = make_subplots(rows=2, cols=2, subplot_titles=("Topo 1", "Topo 2", "Topo 3", "Topo 4"), horizontal_spacing=0.05, vertical_spacing=0.1)
fig_e.add_trace(get_topo(z1), row=1, col=1)
fig_e.add_trace(get_topo(z2), row=1, col=2)
fig_e.add_trace(get_topo(z3), row=2, col=1)
fig_e.add_trace(get_topo(z4), row=2, col=2)
fig_e.update_layout(**layout_update, title="E: Component Loadings (Topographies)")
fig_e.update_xaxes(showgrid=False, zeroline=False, showticklabels=False)
fig_e.update_yaxes(showgrid=False, zeroline=False, showticklabels=False)

# Convert to HTML
html_a = fig_a.to_html(full_html=False, include_plotlyjs='cdn')
html_b = fig_b.to_html(full_html=False, include_plotlyjs=False)
html_c = fig_c.to_html(full_html=False, include_plotlyjs=False)
html_d = fig_d.to_html(full_html=False, include_plotlyjs=False)
html_e = fig_e.to_html(full_html=False, include_plotlyjs=False)

# ==========================================
# 3. BUILD THE HTML GRID LAYOUT
# ==========================================
html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <title>BCI Academic Visuals</title>
    <style>
        body {{ font-family: 'Segoe UI', sans-serif; background-color: #0f172a; color: #f8fafc; padding: 20px; margin: 0; }}
        .dashboard-grid {{
            display: grid;
            grid-template-columns: 1fr 1fr;
            grid-template-rows: auto auto auto;
            gap: 20px;
            max-width: 1400px;
            margin: auto;
        }}
        .card {{ background-color: #1e293b; border-radius: 12px; padding: 15px; border: 1px solid #334155; }}
        .panel-a {{ grid-column: 1; grid-row: 1; }}
        .panel-b {{ grid-column: 2; grid-row: 1; }}
        .panel-c {{ grid-column: 1; grid-row: 2; }}
        .panel-d {{ grid-column: 1; grid-row: 3; }}
        .panel-e {{ grid-column: 2; grid-row: 2 / span 2; display: flex; flex-direction: column; justify-content: center; }}
        .header {{ max-width: 1400px; margin: auto; margin-bottom: 20px; border-bottom: 2px solid #0284c7; padding-bottom: 10px; display: flex; justify-content: space-between; }}
        h1 {{ color: #38bdf8; margin: 0; }}
        .student-info {{ text-align: right; color: #94a3b8; font-size: 14px; }}
    </style>
</head>
<body>
    <div class="header">
        <h1>🧠 BCI Neuro-Visual Analysis Dashboard</h1>
        <div class="student-info">
            <strong>Practical Assessment: Test 3</strong><br>
            Student: Banknote | ID: 12345678
        </div>
    </div>

    <div class="dashboard-grid">
        <div class="card panel-a">{html_a}</div>
        <div class="card panel-b">{html_b}</div>
        <div class="card panel-c">{html_c}</div>
        <div class="card panel-d">{html_d}</div>
        <div class="card panel-e">{html_e}</div>
    </div>
</body>
</html>
"""

# Save HTML File
os.makedirs("data", exist_ok=True)
with open("data/bci_dashboard.html", "w", encoding="utf-8") as f:
    f.write(html_content)

# ==========================================
# 4. START LOCAL SERVER
# ==========================================
PORT = 8009

class QuietHandler(http.server.SimpleHTTPRequestHandler):
    def log_message(self, format, *args): pass 

def start_server():
    with socketserver.TCPServer(("", PORT), QuietHandler) as httpd:
        httpd.serve_forever()

try:
    import urllib.request
    urllib.request.urlopen(f"http://localhost:{PORT}/", timeout=1)
except:
    thread = threading.Thread(target=start_server)
    thread.daemon = True
    thread.start()

url = f"http://localhost:{PORT}/data/bci_dashboard.html"
print(f"✅ Dashboard running on port {PORT}!")
display.display(display.HTML(f'<h3>👉 <a href="{url}" target="_blank" style="color: #38bdf8; font-size: 24px; text-decoration: underline;">Click Here to Open Your Dashboard</a></h3>'))

⏳ Building Publication-Style Dashboard (No Statsmodels required)...
✅ Dashboard running on port 8009!


In [9]:
import os
import http.server
import socketserver
import threading
import IPython.display as display
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import numpy as np
from plotly.subplots import make_subplots

print("⏳ Building Final Tabbed Dashboard for Ankita Mandlik...")

# ==========================================
# 1. GENERATE DATA FOR ALL TABS
# ==========================================
np.random.seed(42)
n_points = 80

# --- Panel A: PCA Scree ---
pca_var = [32, 9, 6, 4, 3.5, 3, 2.5, 2, 1.5, 1]
pca_x = [f"PC{i}" for i in range(1, 11)]

# --- Scatter Data & Manual Regression ---
x_gaf = np.random.normal(0, 20, n_points)
y_panss = -0.6 * x_gaf + np.random.normal(0, 10, n_points)
x_panss = np.random.normal(0, 20, n_points)

def get_regression_line(x, y):
    m, b = np.polyfit(x, y, 1)
    x_line = np.linspace(min(x), max(x), 100)
    y_line = m * x_line + b
    return x_line, y_line

x_line_b, y_line_b = get_regression_line(x_gaf, y_panss)
x_line_c, y_line_c = get_regression_line(x_gaf, np.random.normal(0, 10, n_points))
x_line_d, y_line_d = get_regression_line(x_panss, np.random.normal(0, 10, n_points))

# --- Panel E: Topographies ---
def create_head_grid():
    x, y = np.linspace(-1, 1, 100), np.linspace(-1, 1, 100)
    X, Y = np.meshgrid(x, y)
    mask = X**2 + Y**2 <= 1
    return X, Y, mask

X, Y, head_mask = create_head_grid()

def get_topo(z_data):
    z_data[~head_mask] = np.nan
    return go.Heatmap(x=X[0], y=Y[:,0], z=z_data, colorscale='RdBu_r', showscale=False, zmin=-1, zmax=1)

z1 = np.exp(-((X - 0.5)**2 + Y**2) / 0.1) - np.exp(-((X + 0.5)**2 + Y**2) / 0.1)
z2 = -np.exp(-((X - 0.5)**2 + Y**2) / 0.1) + np.exp(-((X + 0.5)**2 + Y**2) / 0.1)
z3 = np.exp(-(X**2 + (Y - 0.6)**2) / 0.1) - np.exp(-(X**2 + (Y + 0.6)**2) / 0.1)
z4 = -np.exp(-(X**2 + (Y - 0.6)**2) / 0.1) + np.exp(-(X**2 + (Y + 0.6)**2) / 0.1)

# --- VR & tDCS Data ---
time_ms = np.linspace(0, 5000, 100)
erd = 15 + 25 * np.exp(-((time_ms - 2500)**2) / (2 * 800**2))
glow = np.clip((erd - 10) / 30, 0, 1) 

df_tdcs = pd.DataFrame({"Condition": ["Sham tDCS", "Anodal tDCS"], "ITR": [12.5, 22.1], "F1": [0.72, 0.85]})

# ==========================================
# 2. BUILD THE PLOTLY FIGURES
# ==========================================
layout_update = dict(template="plotly_dark", paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', margin=dict(l=40, r=20, t=40, b=40))

# Tab 1: PCA & Scatter Grid
fig_a = go.Figure(go.Scatter(x=pca_x, y=pca_var, mode='lines+markers', line=dict(color='#f472b6', width=3), marker=dict(size=8, color='#38bdf8')))
fig_a.update_layout(**layout_update, title="A: PCA Scree Plot", xaxis_title="Component", yaxis_title="Variance (%)")

def build_scatter(x, y, x_line, y_line, title, color, x_label, y_label):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x, y=y, mode='markers', marker=dict(color=color, size=8, opacity=0.6), name='Data'))
    fig.add_trace(go.Scatter(x=x_line, y=y_line, mode='lines', line=dict(color='#f472b6', width=2), name='Regression'))
    fig.update_layout(**layout_update, title=title, xaxis_title=x_label, yaxis_title=y_label)
    return fig

fig_b = build_scatter(x_gaf, y_panss, x_line_b, y_line_b, "B: GAF vs PANSS", '#38bdf8', "GAF", "PANSS")
fig_c = build_scatter(x_gaf, np.random.normal(0, 10, n_points), x_line_c, y_line_c, "C: PCA 1 vs GAF", '#38bdf8', "GAF", "PCA 1")
fig_d = build_scatter(x_panss, np.random.normal(0, 10, n_points), x_line_d, y_line_d, "D: PCA 2 vs PANSS", '#c084fc', "PANSS", "PCA 2")

# Tab 2: Topographies
fig_e = make_subplots(rows=2, cols=2, subplot_titles=("Topo 1", "Topo 2", "Topo 3", "Topo 4"), horizontal_spacing=0.05, vertical_spacing=0.1)
fig_e.add_trace(get_topo(z1), row=1, col=1)
fig_e.add_trace(get_topo(z2), row=1, col=2)
fig_e.add_trace(get_topo(z3), row=2, col=1)
fig_e.add_trace(get_topo(z4), row=2, col=2)
fig_e.update_layout(**layout_update, title="Component Loadings (Topographies)")
fig_e.update_xaxes(showgrid=False, zeroline=False, showticklabels=False)
fig_e.update_yaxes(showgrid=False, zeroline=False, showticklabels=False)

# Tab 3: VR & tDCS
fig_erd = go.Figure()
fig_erd.add_trace(go.Scatter(x=time_ms, y=erd, name="ERD %", line=dict(color='#a855f7', width=2)))
fig_erd.add_trace(go.Scatter(x=time_ms, y=glow*100, name="VR Glow", line=dict(color='#f59e0b', dash='dot', width=2)))
fig_erd.update_layout(**layout_update, title="μ-ERD Neurofeedback & VR Glow", xaxis_title="Time (ms)", yaxis_title="Value (%)")

fig_tdcs = go.Figure()
fig_tdcs.add_trace(go.Bar(x=df_tdcs["Condition"], y=df_tdcs["ITR"], name="ITR (bits/min)", marker_color="#ef4444", text=df_tdcs["ITR"], textposition='auto', width=0.4))
fig_tdcs.add_trace(go.Bar(x=df_tdcs["Condition"], y=df_tdcs["F1"], name="Macro-F1", marker_color="#38bdf8", text=df_tdcs["F1"], textposition='auto', width=0.4))
fig_tdcs.update_layout(**layout_update, barmode='group', title="tDCS ITR & F1 Improvement")

# Convert to HTML strings
html_a = fig_a.to_html(full_html=False, include_plotlyjs='cdn')
html_b = fig_b.to_html(full_html=False, include_plotlyjs=False)
html_c = fig_c.to_html(full_html=False, include_plotlyjs=False)
html_d = fig_d.to_html(full_html=False, include_plotlyjs=False)
html_e = fig_e.to_html(full_html=False, include_plotlyjs=False)
html_erd = fig_erd.to_html(full_html=False, include_plotlyjs=False)
html_tdcs = fig_tdcs.to_html(full_html=False, include_plotlyjs=False)

# ==========================================
# 3. BUILD THE HTML UI WITH TABS
# ==========================================
html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <title>BCI Dashboard - Ankita Mandlik</title>
    <style>
        body {{ font-family: 'Segoe UI', sans-serif; background-color: #0f172a; color: #f8fafc; padding: 20px; margin: 0; }}
        .container {{ max-width: 1300px; margin: auto; }}
        
        /* Header */
        .header {{ display: flex; justify-content: space-between; align-items: flex-start; border-bottom: 2px solid #0284c7; padding-bottom: 15px; margin-bottom: 25px; }}
        .header-text h1 {{ color: #38bdf8; margin: 0 0 5px 0; font-size: 28px; }}
        .header-text h2 {{ color: #94a3b8; margin: 0; font-size: 16px; font-weight: normal; }}
        .student-info {{ text-align: right; background-color: #1e293b; padding: 10px 20px; border-radius: 8px; border: 1px solid #334155; }}
        .student-info p {{ margin: 0; font-size: 16px; }}
        .student-info span {{ color: #38bdf8; font-weight: bold; }}
        
        /* Tabs */
        .tab {{ overflow: hidden; border-bottom: 1px solid #334155; margin-bottom: 20px; }}
        .tab button {{ background-color: inherit; float: left; border: none; outline: none; cursor: pointer; padding: 14px 20px; transition: 0.3s; color: #94a3b8; font-size: 16px; font-weight: bold; }}
        .tab button:hover {{ background-color: #1e293b; color: #f8fafc; }}
        .tab button.active {{ background-color: #1e293b; color: #38bdf8; border-bottom: 3px solid #38bdf8; }}
        .tabcontent {{ display: none; animation: fadeEffect 0.5s; }}
        
        /* Grids for panels */
        .grid-2x2 {{ display: grid; grid-template-columns: 1fr 1fr; gap: 20px; }}
        .card {{ background-color: #1e293b; border-radius: 12px; padding: 15px; border: 1px solid #334155; }}
        
        @keyframes fadeEffect {{ from {{opacity: 0;}} to {{opacity: 1;}} }}
    </style>
    <script>
        function openTab(evt, tabName) {{
            var i, tabcontent, tablinks;
            tabcontent = document.getElementsByClassName("tabcontent");
            for (i = 0; i < tabcontent.length; i++) {{ tabcontent[i].style.display = "none"; }}
            tablinks = document.getElementsByClassName("tablinks");
            for (i = 0; i < tablinks.length; i++) {{ tablinks[i].className = tablinks[i].className.replace(" active", ""); }}
            document.getElementById(tabName).style.display = "block";
            evt.currentTarget.className += " active";
        }}
    </script>
</head>
<body>
    <div class="container">
        
        <!-- HEADER -->
        <div class="header">
            <div class="header-text">
                <h1>🧠 BCI Neuro-Visual Analysis Dashboard</h1>
                <h2>Practical Assessment: Test 3 — Advanced Hybrid BCI & Neuro-VR Systems</h2>
            </div>
            <div class="student-info">
                <p>Student: <span>Ankita Mandlik</span></p>
            </div>
        </div>
        
        <!-- TABS -->
        <div class="tab">
            <button class="tablinks active" onclick="openTab(event, 'PCA_ML')">PCA & ML Correlations</button>
            <button class="tablinks" onclick="openTab(event, 'Topo')">Brain Topographies</button>
            <button class="tablinks" onclick="openTab(event, 'VR_tDCS')">VR & tDCS Clinical</button>
        </div>

        <!-- TAB 1: PCA & ML -->
        <div id="PCA_ML" class="tabcontent" style="display: block;">
            <div class="grid-2x2">
                <div class="card">{html_a}</div>
                <div class="card">{html_b}</div>
                <div class="card">{html_c}</div>
                <div class="card">{html_d}</div>
            </div>
        </div>

        <!-- TAB 2: TOPOGRAPHIES -->
        <div id="Topo" class="tabcontent">
            <div class="card">{html_e}</div>
        </div>

        <!-- TAB 3: VR & tDCS -->
        <div id="VR_tDCS" class="tabcontent">
            <div class="grid-2x2">
                <div class="card">{html_erd}</div>
                <div class="card">{html_tdcs}</div>
            </div>
        </div>

    </div>
</body>
</html>
"""

# Save HTML File
os.makedirs("data", exist_ok=True)
with open("data/bci_dashboard.html", "w", encoding="utf-8") as f:
    f.write(html_content)

# ==========================================
# 4. START LOCAL SERVER
# ==========================================
PORT = 8010

class QuietHandler(http.server.SimpleHTTPRequestHandler):
    def log_message(self, format, *args): pass 

def start_server():
    with socketserver.TCPServer(("", PORT), QuietHandler) as httpd:
        httpd.serve_forever()

try:
    import urllib.request
    urllib.request.urlopen(f"http://localhost:{PORT}/", timeout=1)
except:
    thread = threading.Thread(target=start_server)
    thread.daemon = True
    thread.start()

url = f"http://localhost:{PORT}/data/bci_dashboard.html"
print(f"✅ Dashboard running on port {PORT}!")
display.display(display.HTML(f'<h3>👉 <a href="{url}" target="_blank" style="color: #38bdf8; font-size: 24px; text-decoration: underline;">Click Here to Open Ankita's Dashboard</a></h3>'))

SyntaxError: unterminated string literal (detected at line 235) (3986849707.py, line 235)

In [10]:
import os
import http.server
import socketserver
import threading
import IPython.display as display
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import numpy as np
from plotly.subplots import make_subplots

print("⏳ Building Final Tabbed Dashboard for Ankita Mandlik...")

# ==========================================
# 1. GENERATE DATA FOR ALL TABS
# ==========================================
np.random.seed(42)
n_points = 80

# --- Panel A: PCA Scree ---
pca_var = [32, 9, 6, 4, 3.5, 3, 2.5, 2, 1.5, 1]
pca_x = [f"PC{i}" for i in range(1, 11)]

# --- Scatter Data & Manual Regression ---
x_gaf = np.random.normal(0, 20, n_points)
y_panss = -0.6 * x_gaf + np.random.normal(0, 10, n_points)
x_panss = np.random.normal(0, 20, n_points)

def get_regression_line(x, y):
    m, b = np.polyfit(x, y, 1)
    x_line = np.linspace(min(x), max(x), 100)
    y_line = m * x_line + b
    return x_line, y_line

x_line_b, y_line_b = get_regression_line(x_gaf, y_panss)
x_line_c, y_line_c = get_regression_line(x_gaf, np.random.normal(0, 10, n_points))
x_line_d, y_line_d = get_regression_line(x_panss, np.random.normal(0, 10, n_points))

# --- Panel E: Topographies ---
def create_head_grid():
    x, y = np.linspace(-1, 1, 100), np.linspace(-1, 1, 100)
    X, Y = np.meshgrid(x, y)
    mask = X**2 + Y**2 <= 1
    return X, Y, mask

X, Y, head_mask = create_head_grid()

def get_topo(z_data):
    z_data[~head_mask] = np.nan
    return go.Heatmap(x=X[0], y=Y[:,0], z=z_data, colorscale='RdBu_r', showscale=False, zmin=-1, zmax=1)

z1 = np.exp(-((X - 0.5)**2 + Y**2) / 0.1) - np.exp(-((X + 0.5)**2 + Y**2) / 0.1)
z2 = -np.exp(-((X - 0.5)**2 + Y**2) / 0.1) + np.exp(-((X + 0.5)**2 + Y**2) / 0.1)
z3 = np.exp(-(X**2 + (Y - 0.6)**2) / 0.1) - np.exp(-(X**2 + (Y + 0.6)**2) / 0.1)
z4 = -np.exp(-(X**2 + (Y - 0.6)**2) / 0.1) + np.exp(-(X**2 + (Y + 0.6)**2) / 0.1)

# --- VR & tDCS Data ---
time_ms = np.linspace(0, 5000, 100)
erd = 15 + 25 * np.exp(-((time_ms - 2500)**2) / (2 * 800**2))
glow = np.clip((erd - 10) / 30, 0, 1) 

df_tdcs = pd.DataFrame({"Condition": ["Sham tDCS", "Anodal tDCS"], "ITR": [12.5, 22.1], "F1": [0.72, 0.85]})

# ==========================================
# 2. BUILD THE PLOTLY FIGURES
# ==========================================
layout_update = dict(template="plotly_dark", paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', margin=dict(l=40, r=20, t=40, b=40))

# Tab 1: PCA & Scatter Grid
fig_a = go.Figure(go.Scatter(x=pca_x, y=pca_var, mode='lines+markers', line=dict(color='#f472b6', width=3), marker=dict(size=8, color='#38bdf8')))
fig_a.update_layout(**layout_update, title="A: PCA Scree Plot", xaxis_title="Component", yaxis_title="Variance (%)")

def build_scatter(x, y, x_line, y_line, title, color, x_label, y_label):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x, y=y, mode='markers', marker=dict(color=color, size=8, opacity=0.6), name='Data'))
    fig.add_trace(go.Scatter(x=x_line, y=y_line, mode='lines', line=dict(color='#f472b6', width=2), name='Regression'))
    fig.update_layout(**layout_update, title=title, xaxis_title=x_label, yaxis_title=y_label)
    return fig

fig_b = build_scatter(x_gaf, y_panss, x_line_b, y_line_b, "B: GAF vs PANSS", '#38bdf8', "GAF", "PANSS")
fig_c = build_scatter(x_gaf, np.random.normal(0, 10, n_points), x_line_c, y_line_c, "C: PCA 1 vs GAF", '#38bdf8', "GAF", "PCA 1")
fig_d = build_scatter(x_panss, np.random.normal(0, 10, n_points), x_line_d, y_line_d, "D: PCA 2 vs PANSS", '#c084fc', "PANSS", "PCA 2")

# Tab 2: Topographies
fig_e = make_subplots(rows=2, cols=2, subplot_titles=("Topo 1", "Topo 2", "Topo 3", "Topo 4"), horizontal_spacing=0.05, vertical_spacing=0.1)
fig_e.add_trace(get_topo(z1), row=1, col=1)
fig_e.add_trace(get_topo(z2), row=1, col=2)
fig_e.add_trace(get_topo(z3), row=2, col=1)
fig_e.add_trace(get_topo(z4), row=2, col=2)
fig_e.update_layout(**layout_update, title="Component Loadings (Topographies)")
fig_e.update_xaxes(showgrid=False, zeroline=False, showticklabels=False)
fig_e.update_yaxes(showgrid=False, zeroline=False, showticklabels=False)

# Tab 3: VR & tDCS
fig_erd = go.Figure()
fig_erd.add_trace(go.Scatter(x=time_ms, y=erd, name="ERD %", line=dict(color='#a855f7', width=2)))
fig_erd.add_trace(go.Scatter(x=time_ms, y=glow*100, name="VR Glow", line=dict(color='#f59e0b', dash='dot', width=2)))
fig_erd.update_layout(**layout_update, title="μ-ERD Neurofeedback & VR Glow", xaxis_title="Time (ms)", yaxis_title="Value (%)")

fig_tdcs = go.Figure()
fig_tdcs.add_trace(go.Bar(x=df_tdcs["Condition"], y=df_tdcs["ITR"], name="ITR (bits/min)", marker_color="#ef4444", text=df_tdcs["ITR"], textposition='auto', width=0.4))
fig_tdcs.add_trace(go.Bar(x=df_tdcs["Condition"], y=df_tdcs["F1"], name="Macro-F1", marker_color="#38bdf8", text=df_tdcs["F1"], textposition='auto', width=0.4))
fig_tdcs.update_layout(**layout_update, barmode='group', title="tDCS ITR & F1 Improvement")

# Convert to HTML strings
html_a = fig_a.to_html(full_html=False, include_plotlyjs='cdn')
html_b = fig_b.to_html(full_html=False, include_plotlyjs=False)
html_c = fig_c.to_html(full_html=False, include_plotlyjs=False)
html_d = fig_d.to_html(full_html=False, include_plotlyjs=False)
html_e = fig_e.to_html(full_html=False, include_plotlyjs=False)
html_erd = fig_erd.to_html(full_html=False, include_plotlyjs=False)
html_tdcs = fig_tdcs.to_html(full_html=False, include_plotlyjs=False)

# ==========================================
# 3. BUILD THE HTML UI WITH TABS
# ==========================================
html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <title>BCI Dashboard - Ankita Mandlik</title>
    <style>
        body {{ font-family: 'Segoe UI', sans-serif; background-color: #0f172a; color: #f8fafc; padding: 20px; margin: 0; }}
        .container {{ max-width: 1300px; margin: auto; }}
        
        /* Header */
        .header {{ display: flex; justify-content: space-between; align-items: flex-start; border-bottom: 2px solid #0284c7; padding-bottom: 15px; margin-bottom: 25px; }}
        .header-text h1 {{ color: #38bdf8; margin: 0 0 5px 0; font-size: 28px; }}
        .header-text h2 {{ color: #94a3b8; margin: 0; font-size: 16px; font-weight: normal; }}
        .student-info {{ text-align: right; background-color: #1e293b; padding: 10px 20px; border-radius: 8px; border: 1px solid #334155; }}
        .student-info p {{ margin: 0; font-size: 16px; }}
        .student-info span {{ color: #38bdf8; font-weight: bold; }}
        
        /* Tabs */
        .tab {{ overflow: hidden; border-bottom: 1px solid #334155; margin-bottom: 20px; }}
        .tab button {{ background-color: inherit; float: left; border: none; outline: none; cursor: pointer; padding: 14px 20px; transition: 0.3s; color: #94a3b8; font-size: 16px; font-weight: bold; }}
        .tab button:hover {{ background-color: #1e293b; color: #f8fafc; }}
        .tab button.active {{ background-color: #1e293b; color: #38bdf8; border-bottom: 3px solid #38bdf8; }}
        .tabcontent {{ display: none; animation: fadeEffect 0.5s; }}
        
        /* Grids for panels */
        .grid-2x2 {{ display: grid; grid-template-columns: 1fr 1fr; gap: 20px; }}
        .card {{ background-color: #1e293b; border-radius: 12px; padding: 15px; border: 1px solid #334155; }}
        
        @keyframes fadeEffect {{ from {{opacity: 0;}} to {{opacity: 1;}} }}
    </style>
    <script>
        function openTab(evt, tabName) {{
            var i, tabcontent, tablinks;
            tabcontent = document.getElementsByClassName("tabcontent");
            for (i = 0; i < tabcontent.length; i++) {{ tabcontent[i].style.display = "none"; }}
            tablinks = document.getElementsByClassName("tablinks");
            for (i = 0; i < tablinks.length; i++) {{ tablinks[i].className = tablinks[i].className.replace(" active", ""); }}
            document.getElementById(tabName).style.display = "block";
            evt.currentTarget.className += " active";
        }}
    </script>
</head>
<body>
    <div class="container">
        
        <!-- HEADER -->
        <div class="header">
            <div class="header-text">
                <h1>🧠 BCI Neuro-Visual Analysis Dashboard</h1>
                <h2>Practical Assessment: Test 3 — Advanced Hybrid BCI & Neuro-VR Systems</h2>
            </div>
            <div class="student-info">
                <p>Student: <span>Ankita Mandlik</span></p>
            </div>
        </div>
        
        <!-- TABS -->
        <div class="tab">
            <button class="tablinks active" onclick="openTab(event, 'PCA_ML')">PCA & ML Correlations</button>
            <button class="tablinks" onclick="openTab(event, 'Topo')">Brain Topographies</button>
            <button class="tablinks" onclick="openTab(event, 'VR_tDCS')">VR & tDCS Clinical</button>
        </div>

        <!-- TAB 1: PCA & ML -->
        <div id="PCA_ML" class="tabcontent" style="display: block;">
            <div class="grid-2x2">
                <div class="card">{html_a}</div>
                <div class="card">{html_b}</div>
                <div class="card">{html_c}</div>
                <div class="card">{html_d}</div>
            </div>
        </div>

        <!-- TAB 2: TOPOGRAPHIES -->
        <div id="Topo" class="tabcontent">
            <div class="card">{html_e}</div>
        </div>

        <!-- TAB 3: VR & tDCS -->
        <div id="VR_tDCS" class="tabcontent">
            <div class="grid-2x2">
                <div class="card">{html_erd}</div>
                <div class="card">{html_tdcs}</div>
            </div>
        </div>

    </div>
</body>
</html>
"""

# Save HTML File
os.makedirs("data", exist_ok=True)
with open("data/bci_dashboard.html", "w", encoding="utf-8") as f:
    f.write(html_content)

# ==========================================
# 4. START LOCAL SERVER
# ==========================================
PORT = 8010

class QuietHandler(http.server.SimpleHTTPRequestHandler):
    def log_message(self, format, *args): pass 

def start_server():
    with socketserver.TCPServer(("", PORT), QuietHandler) as httpd:
        httpd.serve_forever()

try:
    import urllib.request
    urllib.request.urlopen(f"http://localhost:{PORT}/", timeout=1)
except:
    thread = threading.Thread(target=start_server)
    thread.daemon = True
    thread.start()

url = f"http://localhost:{PORT}/data/bci_dashboard.html"
print(f"✅ Dashboard running on port {PORT}!")
# FIXED THE APOSTROPHE HERE
display.display(display.HTML(f'<h3>👉 <a href="{url}" target="_blank" style="color: #38bdf8; font-size: 24px; text-decoration: underline;">Click Here to Open Ankita Mandlik Dashboard</a></h3>'))

⏳ Building Final Tabbed Dashboard for Ankita Mandlik...
✅ Dashboard running on port 8010!


In [11]:
import os
import http.server
import socketserver
import threading
import IPython.display as display
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import numpy as np
from plotly.subplots import make_subplots

print("⏳ Building Advanced Scientific Visuals for Ankita Mandlik...")

# ==========================================
# 1. GENERATE DATA FOR THE VISUALS
# ==========================================
# --- Fig 1: Grid of Confusion Matrices ---
algorithms = ["k-Nearest Neighbors", "Logistic Regression", "Naive Bayes", 
              "Support Vector Machine", "Decision Tree", "Random Forest", 
              "Gradient Boosting", "Multi-Layer Perceptron", "Deep Neural Network"]
# Simulate slightly different confusion matrices for each
cms = []
for i in range(9):
    base = np.array([[28, 0, 0], [3, 17, 0], [0, 0, 19]])
    if i % 3 == 0: base[0,0] -= np.random.randint(0,2) # Add small variations
    if i % 3 == 1: base[1,1] += np.random.randint(-1,2)
    cms.append(base)

fig_cm_grid = make_subplots(rows=3, cols=3, subplot_titles=algorithms, horizontal_spacing=0.1, vertical_spacing=0.15)
for i, cm in enumerate(cms):
    row = i // 3 + 1
    col = i % 3 + 1
    fig_cm_grid.add_trace(go.Heatmap(z=cm, text=cm, texttemplate="%{text}", showscale=(i==2), colorscale='Reds', x=['Adelie', 'Chinstrap', 'Gentoo'], y=['Adelie', 'Chinstrap', 'Gentoo']), row=row, col=col)
fig_cm_grid.update_layout(height=700, template="plotly_dark", title="Confusion Matrix using varied algorithms", showlegend=False)

# --- Fig 2: Spectrograms (Time-Frequency) ---
time = np.linspace(-2, 6, 100)
freq = np.linspace(0, 60, 60)
T, F = np.meshgrid(time, freq)
# Simulate stimulus artifact and ERD/ERS
Z_spec = np.zeros_like(T)
Z_spec += np.exp(-((T-0)**2 + (F-10)**2)/2) * 2  # Artifact
Z_spec -= np.exp(-((T-2)**2 + (F-10)**2)/8)      # Alpha ERD
Z_spec += np.exp(-((T-3)**2 + (F-25)**2)/8)      # Beta ERS

fig_spec = go.Figure(go.Heatmap(z=Z_spec, x=time, y=freq, colorscale='Jet', zmin=-1, zmax=2))
fig_spec.update_layout(title="Spectrograms of Test-Stimulus Event-Related Responses", xaxis_title="Time (s)", yaxis_title="Frequency (Hz)", template="plotly_dark", height=400)

# --- Fig 3: 3D Brain-like Topographies ---
def get_mesh_data():
    x = np.linspace(-1, 1, 50)
    y = np.linspace(-1, 1, 50)
    X, Y = np.meshgrid(x, y)
    Z = np.exp(-((X-0.3)**2 + Y**2)/0.2) - np.exp(-((X+0.3)**2 + Y**2)/0.2)
    Z[X**2 + Y**2 > 1] = np.nan
    return X, Y, Z

X, Y, Z = get_mesh_data()
fig_3d = go.Figure(go.Surface(x=X, y=Y, z=Z, colorscale='RdBu_r', showscale=False))
fig_3d.update_layout(title="Event-related Spectral Power Changes", template="plotly_dark", height=500, scene=dict(xaxis=dict(showticklabels=False), yaxis=dict(showticklabels=False), zaxis=dict(showticklabels=False)))

# --- Fig 4: Multi-Bar Statistical Plot ---
conditions = ['Left', 'Right', 'Sham']
rt = [625, 615, 608]
acc = [90.5, 93.0, 92.5]
err_rt = [15, 12, 14]
err_acc = [2, 1.5, 1.8]

fig_bars = make_subplots(rows=1, cols=2, subplot_titles=("a) Mean RT (ms)", "b) Mean Accuracy (%)"))
fig_bars.add_trace(go.Bar(x=conditions, y=rt, error_y=dict(type='data', array=err_rt), marker_color=['#ff6b6b', '#00d2d3', '#c8d6e5'], text=rt, textposition='auto', name="RT"), row=1, col=1)
fig_bars.add_trace(go.Bar(x=conditions, y=acc, error_y=dict(type='data', array=err_acc), marker_color=['#ff6b6b', '#00d2d3', '#c8d6e5'], text=acc, textposition='auto', name="Acc"), row=1, col=2)
fig_bars.update_layout(template="plotly_dark", height=400, showlegend=False, title="Stimulation Effects on Reaction Time and Accuracy")

# ==========================================
# 2. BUILD THE HTML UI
# ==========================================
html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <title>BCI Dashboard - Ankita Mandlik</title>
    <style>
        body {{ font-family: 'Segoe UI', sans-serif; background-color: #0f172a; color: #f8fafc; padding: 20px; margin: 0; }}
        .container {{ max-width: 1300px; margin: auto; }}
        .header {{ display: flex; justify-content: space-between; align-items: flex-start; border-bottom: 2px solid #0284c7; padding-bottom: 15px; margin-bottom: 25px; }}
        .header-text h1 {{ color: #38bdf8; margin: 0 0 5px 0; font-size: 28px; }}
        .header-text h2 {{ color: #94a3b8; margin: 0; font-size: 16px; font-weight: normal; }}
        .student-info {{ text-align: right; background-color: #1e293b; padding: 10px 20px; border-radius: 8px; border: 1px solid #334155; }}
        .student-info p {{ margin: 0; font-size: 16px; }}
        .student-info span {{ color: #38bdf8; font-weight: bold; }}
        .tab {{ overflow: hidden; border-bottom: 1px solid #334155; margin-bottom: 20px; }}
        .tab button {{ background-color: inherit; float: left; border: none; outline: none; cursor: pointer; padding: 14px 20px; transition: 0.3s; color: #94a3b8; font-size: 16px; font-weight: bold; }}
        .tab button:hover {{ background-color: #1e293b; color: #f8fafc; }}
        .tab button.active {{ background-color: #1e293b; color: #38bdf8; border-bottom: 3px solid #38bdf8; }}
        .tabcontent {{ display: none; animation: fadeEffect 0.5s; }}
        .grid-2 {{ display: grid; grid-template-columns: 1fr 1fr; gap: 20px; }}
        .card {{ background-color: #1e293b; border-radius: 12px; padding: 15px; border: 1px solid #334155; }}
        @keyframes fadeEffect {{ from {{opacity: 0;}} to {{opacity: 1;}} }}
    </style>
    <script>
        function openTab(evt, tabName) {{
            var i, tabcontent, tablinks;
            tabcontent = document.getElementsByClassName("tabcontent");
            for (i = 0; i < tabcontent.length; i++) {{ tabcontent[i].style.display = "none"; }}
            tablinks = document.getElementsByClassName("tablinks");
            for (i = 0; i < tablinks.length; i++) {{ tablinks[i].className = tablinks[i].className.replace(" active", ""); }}
            document.getElementById(tabName).style.display = "block";
            evt.currentTarget.className += " active";
        }}
    </script>
</head>
<body>
    <div class="container">
        <div class="header">
            <div class="header-text">
                <h1>🧠 Advanced Neuro-Visual Dashboard</h1>
                <h2>Practical Assessment: Test 3 — Advanced Hybrid BCI & Neuro-VR Systems</h2>
            </div>
            <div class="student-info">
                <p>Student: <span>Ankita Mandlik</span></p>
            </div>
        </div>
        
        <div class="tab">
            <button class="tablinks active" onclick="openTab(event, 'CM_Grid')">1. Confusion Matrices</button>
            <button class="tablinks" onclick="openTab(event, 'Spectro')">2. Time-Frequency Spectrograms</button>
            <button class="tablinks" onclick="openTab(event, 'Brain3D')">3. 3D Cortical Activations</button>
            <button class="tablinks" onclick="openTab(event, 'Stats')">4. tDCS Statistical Effects</button>
        </div>

        <!-- TAB 1 -->
        <div id="CM_Grid" class="tabcontent" style="display: block;">
            <div class="card">{fig_cm_grid.to_html(full_html=False, include_plotlyjs='cdn')}</div>
        </div>

        <!-- TAB 2 -->
        <div id="Spectro" class="tabcontent">
            <div class="card">{fig_spec.to_html(full_html=False, include_plotlyjs=False)}</div>
        </div>

        <!-- TAB 3 -->
        <div id="Brain3D" class="tabcontent">
            <div class="card">{fig_3d.to_html(full_html=False, include_plotlyjs=False)}</div>
        </div>

        <!-- TAB 4 -->
        <div id="Stats" class="tabcontent">
            <div class="card">{fig_bars.to_html(full_html=False, include_plotlyjs=False)}</div>
        </div>
    </div>
</body>
</html>
"""

# Save HTML File
os.makedirs("data", exist_ok=True)
with open("data/bci_dashboard.html", "w", encoding="utf-8") as f:
    f.write(html_content)

# ==========================================
# 3. START LOCAL SERVER
# ==========================================
PORT = 8011

class QuietHandler(http.server.SimpleHTTPRequestHandler):
    def log_message(self, format, *args): pass 

def start_server():
    with socketserver.TCPServer(("", PORT), QuietHandler) as httpd:
        httpd.serve_forever()

try:
    import urllib.request
    urllib.request.urlopen(f"http://localhost:{PORT}/", timeout=1)
except:
    thread = threading.Thread(target=start_server)
    thread.daemon = True
    thread.start()

url = f"http://localhost:{PORT}/data/bci_dashboard.html"
print(f"✅ Dashboard running on port {PORT}!")
display.display(display.HTML(f'<h3>👉 <a href="{url}" target="_blank" style="color: #38bdf8; font-size: 24px; text-decoration: underline;">Click Here to Open the Advanced Scientific Dashboard</a></h3>'))

⏳ Building Advanced Scientific Visuals for Ankita Mandlik...
✅ Dashboard running on port 8011!
